# Drive 체크포인트 정리 (`outputs/` + `sweep/`)

학습 중간에 저장된 epoch별 체크포인트 (`*.pt`, `project_z_*.npy`, `company_z_*.npy`, `metrics_*_epoch*.json`) 중 **불필요한 중간 파일**을 Drive에서 삭제합니다.

## 두 가지 모드

| Mode | 설명 | 권장 폴더 |
|---|---|---|
| `--keep-mode explicit` (기본) | 스크립트의 `KEEP` 딕셔너리에 명시된 peak/final epoch만 보존 | `outputs/` (장시간 single-cell 실험) |
| `--keep-mode last-epoch` | 각 tag별로 **마지막 epoch만** 보존 | `sweep/` (각 cell이 동일 epochs까지 돌리는 grid) |

tag-level 파일(`history_*.json`, `metrics_*.json`, `metrics_*_clean.json`)은 두 모드 모두 자동 보존.

## 안전장치

1. `--dry-run` 또는 `--execute` 명시 필수
2. unknown tag는 자동 UNTOUCHED → 의외의 파일 보호
3. Drive 휴지통 **30일** 보관 → 잘못 지웠어도 복구 가능

## 사용 순서

1. **Mount + Pull repo** (Step 0)
2. **outputs/ 정리** (Step 1-3): `keep-mode=explicit`
3. **sweep/ 정리** (Step 4-6): `keep-mode=last-epoch --recursive`

## Step 0. Drive mount + 저장소 동기화

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_DIR = '/content/apollo.M1.GNN'
GIT_URL = 'https://github.com/osung/apollo.M1.GNN.git'

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git pull origin main
else:
    !git clone $GIT_URL $REPO_DIR
    %cd $REPO_DIR

---

## A. `outputs/` 정리 — `keep-mode=explicit`

스크립트의 `KEEP` 딕셔너리에 명시된 peak/final epoch만 보존. 장시간 single-cell 학습 결과(GFM h128 l2 sim=10 ep200..300 등)에 적합.

**다른 계정에서 사용하려면** 아래 `OUTPUTS_DIR`을 변경하세요.

### Step 1. 현재 outputs/ 상태

In [ ]:
OUTPUTS_DIR = '/content/drive/MyDrive/apollo.M1.GNN/outputs'

import os, subprocess
if not os.path.isdir(OUTPUTS_DIR):
    print(f'[WARN] {OUTPUTS_DIR} 가 존재하지 않습니다.')
else:
    n = len(os.listdir(OUTPUTS_DIR))
    sz = subprocess.check_output(['du', '-sh', OUTPUTS_DIR], text=True).split()[0]
    print(f'경로: {OUTPUTS_DIR}\n파일 수: {n}\n총 크기: {sz}')

### Step 2. DRY-RUN (outputs)

In [ ]:
!python scripts/cleanup_drive_outputs.py --outputs-dir "$OUTPUTS_DIR" --keep-mode explicit --dry-run

### Step 3. EXECUTE (outputs)

In [ ]:
!python scripts/cleanup_drive_outputs.py --outputs-dir "$OUTPUTS_DIR" --keep-mode explicit --execute

---

## B. `sweep/` 정리 — `keep-mode=last-epoch --recursive`

각 sweep cell마다 ep20/40/.../ep_total 체크포인트가 쌓여있고, 일반적으로 **마지막 epoch만 의미가 있음** (중간은 진행 중 실패 진단용 정도). 각 tag별로 highest epoch만 남기고 모두 삭제.

`sweep/` 루트는 `gfm/`, `gcn/`, `sage/` 같은 모델별 서브폴더로 나뉘어 있으므로 `--recursive`로 일괄 처리.

### Step 4. 현재 sweep/ 상태

In [ ]:
SWEEP_DIR = '/content/drive/MyDrive/apollo.M1.GNN/sweep'

import os, subprocess
if not os.path.isdir(SWEEP_DIR):
    print(f'[WARN] {SWEEP_DIR} 가 존재하지 않습니다.')
else:
    sz = subprocess.check_output(['du', '-sh', SWEEP_DIR], text=True).split()[0]
    n = subprocess.check_output(
        f'find "{SWEEP_DIR}" -type f | wc -l', shell=True, text=True
    ).strip()
    print(f'경로: {SWEEP_DIR}\n파일 수 (recursive): {n}\n총 크기: {sz}')
    print('\n서브폴더:')
    for d in sorted(os.listdir(SWEEP_DIR)):
        p = os.path.join(SWEEP_DIR, d)
        if os.path.isdir(p):
            d_sz = subprocess.check_output(['du', '-sh', p], text=True).split()[0]
            d_n = len(os.listdir(p))
            print(f'  {d:<10} {d_n:>5} files  {d_sz}')

### Step 5. DRY-RUN (sweep, recursive)

In [ ]:
!python scripts/cleanup_drive_outputs.py \
    --outputs-dir "$SWEEP_DIR" \
    --keep-mode last-epoch --recursive --dry-run

### Step 6. EXECUTE (sweep, recursive)

In [ ]:
!python scripts/cleanup_drive_outputs.py \
    --outputs-dir "$SWEEP_DIR" \
    --keep-mode last-epoch --recursive --execute

## (선택) 특정 모델만 정리

예를 들어 `sweep/gfm/`만 정리하려면:

```bash
!python scripts/cleanup_drive_outputs.py \
    --outputs-dir /content/drive/MyDrive/apollo.M1.GNN/sweep/gfm \
    --keep-mode last-epoch --dry-run
```

## 결과 확인

In [ ]:
import subprocess, os
for label, path in [('outputs', OUTPUTS_DIR), ('sweep', SWEEP_DIR)]:
    if os.path.isdir(path):
        sz = subprocess.check_output(['du', '-sh', path], text=True).split()[0]
        n = subprocess.check_output(
            f'find "{path}" -type f | wc -l', shell=True, text=True
        ).strip()
        print(f'{label:<8}: {n:>6} files  {sz}')